# LoRA training experiments

One notebook run = one experiment config.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, subprocess, sys
from pathlib import Path

def pip(args):
    print('pip', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip'] + args)

def tv_pkg():
    import torch
    v = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
    return {'2.8': 'torchvision==0.23.0', '2.9': 'torchvision==0.24.0', '2.10': 'torchvision==0.25.0', '2.11': 'torchvision==0.26.0'}.get(v, 'torchvision')

marker = Path('/content/drive/MyDrive/experiments_vbert/.deps_train_h3_l2')

if not marker.exists():
    pip(['uninstall', '-y', 'torchao'])
    pkgs = [
        'numpy==1.26.4', 'pandas==2.2.2', 'datasets==3.6.0', 'Pillow==11.3.0', 'tqdm==4.67.1', 'pyarrow==19.0.1', 'safetensors==0.5.3', 'fsspec[http]==2025.3.0', 'huggingface_hub>=1.5.0,<2.0', 'tokenizers>=0.22.0,<=0.23.0', 'requests==2.32.4', 'peft'
    ]
    pip(['install', '-q', '--no-cache-dir', '--force-reinstall'] + pkgs)
    pip(['install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', 'git+https://github.com/huggingface/transformers'])
    pip(['install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', tv_pkg()])
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text('ok', encoding='utf-8')
    os.kill(os.getpid(), 9)

print('deps ok')

deps ok


In [ ]:
from pathlib import Path
import json, math, random, tarfile, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, load_from_disk
from PIL import Image
from tqdm.auto import tqdm
from peft import LoraConfig, get_peft_model
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer, Idefics3ImageProcessor, ModernVBertConfig, ColModernVBertConfig, ColModernVBertForRetrieval, ColModernVBertProcessor

In [ ]:
SEED = 42
BASE_MODEL = 'ModernVBERT/colmodernvbert-merged'
EXPERIMENT_ID = 'RU-LoRA-R8-L2'
RUN_ID = time.strftime('%Y%m%d_%H%M%S')
OUT = Path('/content/drive/MyDrive/experiments_vbert/training_runs') / EXPERIMENT_ID / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

VIDORE_TAR = Path('/content/drive/MyDrive/vidore_ru_result.tar')
VIDORE_ROOT = Path('/content/vidore_ru_result')
if not VIDORE_ROOT.exists():
    VIDORE_ROOT.mkdir(parents=True)
    with tarfile.open(VIDORE_TAR) as tar:
        tar.extractall(VIDORE_ROOT)

def found(paths):
    return sorted(paths, key=lambda p: len(p.parts))[0]

def hf_dir(name):
    hits = [p for p in VIDORE_ROOT.rglob(name) if p.is_dir() and (p / 'state.json').exists()]
    return found(hits or [p for p in VIDORE_ROOT.rglob(name) if p.is_dir()])

IMG_CACHE = {}

def fix_img(x):
    if not isinstance(x, str):
        return x
    p = Path(x)
    if p.exists():
        return str(p)

    parts = p.parts
    tries = [VIDORE_ROOT / p, Path('/content') / p, Path('/content/drive/MyDrive') / p]
    if parts and parts[0] == 'vidore_ru_work':
        tries.append(VIDORE_ROOT / Path(*parts[1:]))
    if 'ru_images' in parts:
        tries.append(VIDORE_ROOT / Path(*parts[parts.index('ru_images'):]))
    for q in tries:
        if q.exists():
            return str(q)

    if x not in IMG_CACHE:
        tail3 = '/'.join(parts[-3:])
        tail2 = '/'.join(parts[-2:])
        hits = [h for h in VIDORE_ROOT.rglob(p.name) if h.as_posix().endswith(tail3) or h.as_posix().endswith(tail2)]
        IMG_CACHE[x] = str(sorted(hits, key=lambda h: len(h.parts))[0]) if hits else x
    return IMG_CACHE[x]

def open_img(x):
    return x.convert('RGB') if isinstance(x, Image.Image) else Image.open(fix_img(x)).convert('RGB')

VIDORE_TRAIN_DIR = hf_dir('train_merged')
VIDORE_VAL_DIR = hf_dir('test_vidore_docvqa_test_subsampled')
VIDORE_TRAIN_JSONL = found(VIDORE_ROOT.rglob('train.jsonl'))
VIDORE_VAL_JSONL = found(VIDORE_ROOT.rglob('test_vidore_docvqa_test_subsampled.jsonl'))
COCO_ID = 'romrawinjp/multilingual-coco'
COCO_TEST_FRACTION = 0.005
COCO_VAL_FRACTION = 0.002
COCO_TRAIN_FRACTION = 0.01
VIDORE_WEIGHT = 0.5
COCO_WEIGHT = 0.5

LR = 3e-4
LORA_RANK = 16
LORA_ALPHA = 32
MICRO_BATCH = 2
GRAD_ACCUM = 8
TRAIN_STEPS = 800
EVAL_EVERY = 100
IMAGE_BATCH_SIZE = 2
QUERY_BATCH_SIZE = 8
TOPK = 20

TRAINABLE_SETUP = 'L2'  # L1 / L2 / L3 / L4
SMOKE_TEST = False
SMOKE_STEPS = 2

random.seed(SEED)
torch.manual_seed(SEED)

/tmp/ipykernel_12717/3836762505.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(VIDORE_ROOT)


In [ ]:
def to_dev(batch, dev):
    return {k: v.to(dev) if hasattr(v, 'to') else v for k, v in batch.items()}

def remap_keys(sd):
    out = {}
    for k, v in sd.items():
        if k.startswith('model.vision_model.vision_model.'):
            k = k.replace('model.vision_model.vision_model.', 'vlm.vision_model.', 1)
        elif k.startswith('model.text_model.'):
            k = k.replace('model.text_model.', 'vlm.text_model.', 1)
        elif k.startswith('model.connector.'):
            k = k.replace('model.connector.', 'vlm.connector.', 1)
        elif k.startswith('custom_text_proj.'):
            k = k.replace('custom_text_proj.', 'embedding_proj_layer.', 1)
        out[k] = v
    return out

def make_processor(mid):
    image_processor = Idefics3ImageProcessor(
        do_convert_rgb=True, do_resize=True, size={'longest_edge': 2048},
        do_image_splitting=True, max_image_size={'longest_edge': 512},
        do_rescale=True, rescale_factor=1/255, do_normalize=True,
        image_mean=[0.5,0.5,0.5], image_std=[0.5,0.5,0.5], do_pad=True,
    )
    tok = AutoTokenizer.from_pretrained(mid)
    return ColModernVBertProcessor(image_processor=image_processor, tokenizer=tok, image_seq_len=64)

def load_base(mid):
    proc = make_processor(mid)
    cfg = ColModernVBertConfig(vlm_config=ModernVBertConfig.from_pretrained(mid), embedding_dim=128)
    model = ColModernVBertForRetrieval(cfg)
    state = load_file(hf_hub_download(mid, 'model.safetensors'))
    missing, unexpected = model.load_state_dict(remap_keys(state), strict=False)
    missing = [x for x in missing if 'position_ids' not in x]
    if missing or unexpected:
        raise RuntimeError(f'bad load: missing={missing[:8]} unexpected={unexpected[:8]}')
    return proc, model

def lora_targets(setup):
    text = ['Wqkv', 'Wo', 'Wi']
    if setup == 'L1':
        return text
    if setup == 'L2':
        return text + ['embedding_proj_layer', 'modality_projection']
    if setup == 'L3':
        return text + ['embedding_proj_layer', 'modality_projection', 'q_proj', 'v_proj']
    return text + ['embedding_proj_layer', 'modality_projection', 'q_proj', 'k_proj', 'v_proj', 'out_proj']

def build_model():
    proc, model = load_base(BASE_MODEL)
    for p in model.parameters():
        p.requires_grad = False
    cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, target_modules=lora_targets(TRAINABLE_SETUP), bias='none')
    model = get_peft_model(model, cfg)
    for name, p in model.named_parameters():
        if any(x in name for x in ['embedding_proj_layer', 'modality_projection']) and TRAINABLE_SETUP in ['L2','L3','L4']:
            p.requires_grad = True
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.bfloat16 if dev == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
    return proc, model.to(device=dev, dtype=dtype), dev

In [ ]:
def vidore_rows(path, limit=None):
    try:
        ds = load_from_disk(str(path))
        raw = [ds[i] for i in range(min(len(ds), limit or len(ds)))]
    except Exception:
        jsonl = VIDORE_TRAIN_JSONL if 'train' in str(path) else VIDORE_VAL_JSONL
        if not jsonl.exists():
            raise
        raw = [json.loads(x) for x in jsonl.read_text(encoding='utf-8').splitlines() if x.strip()]
        raw = raw[:limit] if limit else raw

    rows = []
    for r in raw:
        q = str(r.get('query_ru') or r.get('query') or '').strip()
        img = fix_img(r.get('image_ru_path') or r.get('image') or r.get('original_image_path'))
        if q and img:
            rows.append(dict(query=q, image=img))
    return rows

def coco_rows(part, limit=None):
    ds = load_dataset(COCO_ID, split='train').shuffle(seed=SEED)
    test_n = max(1, int(len(ds) * COCO_TEST_FRACTION))
    val_n = max(1, int(len(ds) * COCO_VAL_FRACTION))
    train_n = max(1, int(len(ds) * COCO_TRAIN_FRACTION))
    spans = {
        'test': (0, test_n),
        'val': (test_n, test_n + val_n),
        'train': (test_n + val_n, test_n + val_n + train_n),
    }
    start, stop = spans[part]
    if limit:
        stop = min(stop, start + limit)
    out = []
    for r in ds.select(range(start, stop)):
        caps = r.get('ru') or []
        if isinstance(caps, str):
            caps = [caps]
        caps = [str(x).strip() for x in caps if str(x).strip()]
        if caps:
            out.append(dict(query=caps[0], image=r['image']))
    return out

def train_pools():
    return {
        'vidore': vidore_rows(VIDORE_TRAIN_DIR, 32 if SMOKE_TEST else None),
        'coco': coco_rows('train', 32 if SMOKE_TEST else None),
    }

def sample_train_batch(pools, size):
    names = ['vidore', 'coco']
    weights = [VIDORE_WEIGHT, COCO_WEIGHT]
    batch = []
    for _ in range(size):
        name = random.choices(names, weights=weights, k=1)[0]
        if not pools[name]:
            name = 'vidore' if pools['vidore'] else 'coco'
        batch.append(random.choice(pools[name]))
    return batch

def val_rows():
    return {
        'vidore_ru_val': vidore_rows(VIDORE_VAL_DIR, 16 if SMOKE_TEST else None),
        'coco_ru_val': coco_rows('val', 16 if SMOKE_TEST else None),
    }

In [ ]:
def split_emb(x, batch):
    mask = batch.get('attention_mask')
    if mask is None:
        return [x[i] for i in range(x.shape[0])]
    return [x[i, mask[i].bool()] for i in range(x.shape[0])]

def score_matrix(proc, q_embs, d_embs):
    return proc.score_retrieval(query_embeddings=q_embs, passage_embeddings=d_embs)

def loss_batch(proc, model, dev, batch_rows):
    queries = [r['query'] for r in batch_rows]
    images = [open_img(r['image']) for r in batch_rows]
    q_batch = to_dev(proc(text=queries, return_tensors='pt'), dev)
    d_batch = to_dev(proc(text=['<image>'] * len(images), images=images, return_tensors='pt'), dev)
    q_emb = split_emb(model(**q_batch).embeddings, q_batch)
    d_emb = split_emb(model(**d_batch).embeddings, d_batch)
    scores = score_matrix(proc, q_emb, d_emb)
    labels = torch.arange(len(batch_rows), device=scores.device)
    return F.cross_entropy(scores, labels)

def ndcg5(rank, good):
    return 1 / math.log2(rank.index(good) + 2) if good in rank[:5] else 0.0

def evaluate(proc, model, dev, rows):
    model.eval()
    q_embs, d_embs = [], []
    for r in rows:
        q = to_dev(proc(text=[r['query']], return_tensors='pt'), dev)
        img = open_img(r['image'])
        d = to_dev(proc(text=['<image>'], images=[img], return_tensors='pt'), dev)
        with torch.inference_mode():
            q_embs += split_emb(model(**q).embeddings, q)
            d_embs += split_emb(model(**d).embeddings, d)
    scores = score_matrix(proc, q_embs, d_embs).detach().cpu()
    out = []
    for i in range(len(rows)):
        order = torch.argsort(scores[i], descending=True).tolist()
        out.append(ndcg5(order, i))
    model.train()
    return float(np.mean(out)) if out else 0.0

In [9]:
proc, model, dev = build_model()
pools = train_pools()
vals = val_rows()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
steps = SMOKE_STEPS if SMOKE_TEST else TRAIN_STEPS
history = []
best = -1.0
best_payload = None

for step in tqdm(range(1, steps + 1), desc='train'):
    batch = sample_train_batch(pools, MICRO_BATCH)
    loss = loss_batch(proc, model, dev, batch) / GRAD_ACCUM
    loss.backward()
    if step % GRAD_ACCUM == 0 or step == steps:
        opt.step(); opt.zero_grad()
    rec = {'step': step, 'loss': float(loss.detach().cpu()) * GRAD_ACCUM}
    if step % (1 if SMOKE_TEST else EVAL_EVERY) == 0 or step == steps:
        v = {name: evaluate(proc, model, dev, data) for name, data in vals.items()}
        mean_score = float(np.mean(list(v.values())))
        rec.update(v); rec['mean_ndcg5'] = mean_score
        if mean_score > best:
            best = mean_score
            best_payload = {'step': step, 'mean_ndcg5': best, **v}
            adapter_dir = OUT / 'adapter'
            model.save_pretrained(adapter_dir)
    history.append(rec)

cfg = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (str, int, float, bool))}
(OUT / 'training_config.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'val_metrics.json').write_text(json.dumps(best_payload or {}, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'best_checkpoint_meta.json').write_text(json.dumps(best_payload or {}, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(history).to_csv(OUT / 'training_history.csv', index=False)
print('saved to', OUT)

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

train:   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(


saved to /content/drive/MyDrive/experiments_vbert/training_runs/RU-LoRA-R8-L2/20260512_102229
